# Практическая работа №1: Введение в SimPy — Моделирование заправочной станции

## Теоретическая справка
SimPy — это библиотека для дискретно-событийного имитационного моделирования на Python. Основные концепции:
- **Environment** — среда симуляции, отслеживающая время и события
- **Process** — генераторная функция, описывающая поведение активного объекта
- **Resource** — ограниченный ресурс (например, заправочная колонка)
- **yield** — приостановка процесса до наступления события (занятия ресурса, задержки)

Время в симуляции абстрактное и измеряется в условных единицах.

In [1]:
# Установка библиотеки (выполнить один раз)
!pip install simpy

In [2]:
import simpy
import random

# Функция-процесс: автомобиль приезжает на АЗС и заправляется
def car(env, name, gas_station, fuel_pump):
    print(f'{env.now:6.2f} | {name} прибыл на АЗС')

    with gas_station.request() as req:
        yield req
        print(f'{env.now:6.2f} | {name} занял колонку')

        fuel_needed = random.randint(20, 40)
        yield fuel_pump.get(fuel_needed)
        yield env.timeout(random.uniform(2, 5))

        print(f'{env.now:6.2f} | {name} заправился ({fuel_needed} л) и уехал')

# Функция-процесс: периодическая доставка топлива
def fuel_truck(env, fuel_pump):
    while True:
        yield env.timeout(10)
        print(f'{env.now:6.2f} | Прибыл бензовоз (+50 л топлива)')
        yield fuel_pump.put(50)

# ✅ НОВАЯ функция: генератор автомобилей с интервалом
def car_generator(env, gas_station, fuel_pump):
    for i in range(4):
        env.process(car(env, f'Автомобиль {i+1}', gas_station, fuel_pump))
        yield env.timeout(3)  # Теперь yield внутри функции!

# Создаём среду симуляции
env = simpy.Environment()

# Создаём ресурсы
gas_station = simpy.Resource(env, capacity=2)
fuel_pump = simpy.Container(env, init=100, capacity=200)

# Запускаем процессы
env.process(fuel_truck(env, fuel_pump))
env.process(car_generator(env, gas_station, fuel_pump))  # ✅ Запускаем генератор

# Запускаем симуляцию
env.run(until=30)

  0.00 | Автомобиль 1 прибыл на АЗС
  0.00 | Автомобиль 1 занял колонку
  3.00 | Автомобиль 2 прибыл на АЗС
  3.00 | Автомобиль 2 занял колонку
  3.26 | Автомобиль 1 заправился (31 л) и уехал
  5.10 | Автомобиль 2 заправился (25 л) и уехал
  6.00 | Автомобиль 3 прибыл на АЗС
  6.00 | Автомобиль 3 занял колонку
  9.00 | Автомобиль 4 прибыл на АЗС
  9.00 | Автомобиль 4 занял колонку
  9.68 | Автомобиль 3 заправился (35 л) и уехал
 10.00 | Прибыл бензовоз (+50 л топлива)
 13.17 | Автомобиль 4 заправился (39 л) и уехал
 20.00 | Прибыл бензовоз (+50 л топлива)


## Задание для самостоятельной работы
Модифицируйте модель:
1. Увеличьте количество колонок до 3
2. Измените интервал прибытия автомобилей на случайный (от 1 до 4 единиц времени)
3. Добавьте вывод статистики в конце симуляции:
   - Сколько автомобилей успело заправиться
   - Сколько топлива осталось в резервуаре
   - Среднее время ожидания колонки (подсказка: сохраняйте время прибытия и время начала заправки)

Реализуйте подсчёт статистики с использованием глобальных переменных или отдельного класса-наблюдателя.

## Контрольные вопросы
1. Что такое дискретно-событийное моделирование и чем оно отличается от непрерывного?
2. Для чего используется оператор `yield` в процессах SimPy?
3. В чём разница между `simpy.Resource` и `simpy.Container`?
4. Почему функции-процессы в SimPy должны быть генераторами (содержать `yield`)?
5. Как задать продолжительность симуляции и что произойдёт после её завершения?

##Самостоятельная работа

In [4]:
import simpy
import random

# Класс для сбора статистики
class Stats:
    def __init__(self):
        self.cars_served = 0
        self.wait_times = []

stats = Stats()


# Функция-процесс: автомобиль приезжает на АЗС и заправляется
def car(env, name, gas_station, fuel_pump):
    arrival_time = env.now
    print(f'{env.now:6.2f} | {name} прибыл на АЗС')

    with gas_station.request() as req:
        yield req

        start_service = env.now
        wait_time = start_service - arrival_time
        stats.wait_times.append(wait_time)

        print(f'{env.now:6.2f} | {name} занял колонку (ждал {wait_time:.2f})')

        fuel_needed = random.randint(20, 40)

        yield fuel_pump.get(fuel_needed)
        yield env.timeout(random.uniform(2, 5))

        stats.cars_served += 1

        print(f'{env.now:6.2f} | {name} заправился ({fuel_needed} л) и уехал')


# Функция-процесс: периодическая доставка топлива
def fuel_truck(env, fuel_pump):
    while True:
        yield env.timeout(10)
        print(f'{env.now:6.2f} | Прибыл бензовоз (+50 л топлива)')
        yield fuel_pump.put(50)


# Генератор автомобилей со случайным интервалом
def car_generator(env, gas_station, fuel_pump):
    for i in range(4):
        env.process(car(env, f'Автомобиль {i+1}', gas_station, fuel_pump))
        # случайный интервал прибытия автомобилей
        yield env.timeout(random.uniform(1, 4))


# Создаём среду симуляции
env = simpy.Environment()

# Ресурсы (увеличено количество колонок до 3)
gas_station = simpy.Resource(env, capacity=3)
fuel_pump = simpy.Container(env, init=100, capacity=200)

# Запускаем процессы
env.process(fuel_truck(env, fuel_pump))
env.process(car_generator(env, gas_station, fuel_pump))

# Запускаем симуляцию
env.run(until=30)


# -------- СТАТИСТИКА --------
print("\n--- Статистика ---")

print(f'Обслужено автомобилей: {stats.cars_served}')
print(f'Топлива осталось: {fuel_pump.level:.2f} л')

if stats.wait_times:
    avg_wait = sum(stats.wait_times) / len(stats.wait_times)
    print(f'Среднее время ожидания колонки: {avg_wait:.2f}')
else:
    print("Нет данных о времени ожидания")

  0.00 | Автомобиль 1 прибыл на АЗС
  0.00 | Автомобиль 1 занял колонку (ждал 0.00)
  2.74 | Автомобиль 1 заправился (35 л) и уехал
  3.49 | Автомобиль 2 прибыл на АЗС
  3.49 | Автомобиль 2 занял колонку (ждал 0.00)
  5.90 | Автомобиль 2 заправился (25 л) и уехал
  6.90 | Автомобиль 3 прибыл на АЗС
  6.90 | Автомобиль 3 занял колонку (ждал 0.00)
  8.89 | Автомобиль 4 прибыл на АЗС
  8.89 | Автомобиль 4 занял колонку (ждал 0.00)
 10.00 | Прибыл бензовоз (+50 л топлива)
 10.93 | Автомобиль 3 заправился (20 л) и уехал
 13.05 | Автомобиль 4 заправился (37 л) и уехал
 20.00 | Прибыл бензовоз (+50 л топлива)

--- Статистика ---
Обслужено автомобилей: 4
Топлива осталось: 83.00 л
Среднее время ожидания колонки: 0.00


##Ответы на вопросы
1. Дискретно-событийное моделирование — система меняется только в моменты событий (например: прибытие машины, начало заправки, приезд бензовоза).  Непрерывное моделирование — состояние системы изменяется постоянно во времени (например: изменение температуры, скорости, давления).

2. yield приостанавливает выполнение процесса, пока не произойдёт определённое событие.

3. Resource — ограниченное количество объектов (например: колонки на АЗС). Container — ограниченное количество вещества или ресурса (например: топливо).

4. Потому что yield позволяет останавливать и возобновлять процесс.
Это нужно, чтобы SimPy мог переключаться между разными процессами и управлять временем симуляции. Также, генераторы позволяют не создавать отдельных глобальных переменных для хранения данных о ресурсах

5. `env.run(until=30)` Это означает, что симуляция будет выполняться до времени 30.
После достижения этого времени выполнение симуляции останавливается, и код продолжает выполняться дальше (например, можно вывести статистику).
